# Shot selection evidence

Final rerun through live80: **512 shots,102 portal-reported shots on target,19.922%** across78 complete saved reports out of80. All102 on-target counters equal102 goals. Two reports lack full shot counters. Distances and keeper save probabilities remain unknown. Historical creation snapshot through live49 is preserved in Git0bfe524.

## Context & Methods

Decision: how to improve attacking shot quality and prevent dangerous opponent chances. Source: `research/v3/matches.json`, extracted from rendered primary portal STATS. The AI overview is excluded. This is saved-report coverage, not the full account history.

### Key Assumptions

A report is the unit of analysis. Missing counters remain missing. Shares use summed numerators and denominators. Recorded setup labels do not prove live execution. Different opponents and HOME/AWAY assignments prevent a causal setup ranking. On-target/goals equality limits interpretation of accuracy and saves. This compact counter audit uses exact-value tables; an aggregate chart would not supply the absent shot distances.

The creation snapshot source is preserved in Git commit `0bfe524` (through live49). Later notebook reruns may read additional reports.

## Data

### Load and validate saved reports

In [1]:
from pathlib import Path
import json, sys
root = Path.cwd()
while not (root / 'research/v3/matches.json').exists():
    if root.parent == root: raise FileNotFoundError('Run from this repository')
    root = root.parent
sys.path.insert(0, str(root / 'research/v3'))
from shots_analysis import analyze
result = analyze(root / 'research/v3/matches.json')
print(json.dumps({'throughReport': result['throughReport'], 'sourceSha256': result['sourceSha256'], **result['coverage']}, indent=2))

{
  "throughReport": "live-80",
  "sourceSha256": "2696e9d18e68df7707b29dab065a2c3a5bf0c27bda7e36d6037b8eb72c7d27d3",
  "recordedReports": 80,
  "completeShotReports": 78,
  "missingShotReports": [
    "live-29",
    "live-72"
  ],
  "scope": "Saved reports only; not the entire account match history"
}


## Results

### Primary portal counters

In [2]:
print(json.dumps(result['allCompleteReports'], indent=2))
assert result['allCompleteReports']['shotsFor'] == result['allCompleteReports']['shotsOnTargetFor'] + result['allCompleteReports']['notMarkedOnTarget']
print('Shot distance result:', result['shotDistanceAnalysis']['successByDistance'])

{
  "reports": 78,
  "shotsFor": 512,
  "shotsOnTargetFor": 102,
  "goalsFor": 102,
  "notMarkedOnTarget": 410,
  "portalOnTargetSharePercent": 19.922,
  "shotsAgainst": 327,
  "shotsOnTargetAgainst": 67,
  "goalsAgainst": 67
}
Shot distance result: None


### Recorded setup comparison

Descriptive only: uncontrolled opponents, small cohorts and uncertain execution prevent a policy-effect claim.

In [3]:
for name in ('A0-M7-formation121', 'A0-M8-formation112'):
    v = result['byRecordedSetup'].get(name)
    if v: print(f"{name}: {v['reports']} reports; {v['shotsOnTargetFor']}/{v['shotsFor']} marked on target ({v['portalOnTargetSharePercent']}%)")
print('Background/unattributed reports:', result['backgroundUnattributedReports']['reports'])

A0-M7-formation121: 8 reports; 10/65 marked on target (15.385%)
A0-M8-formation112: 11 reports; 20/97 marked on target (20.619%)
Background/unattributed reports: 13


### Reconcile consequential totals independently

In [4]:
records = json.loads((root / 'research/v3/matches.json').read_text())['matches']
complete = [r for r in records if all(r.get(k) is not None for k in ('shotsFor', 'shotsAgainst', 'shotsOnTargetFor', 'shotsOnTargetAgainst'))]
shots = target = 0
for row in complete:
    shots += row['shotsFor']
    target += row['shotsOnTargetFor']
assert (shots, target) == (result['allCompleteReports']['shotsFor'], result['allCompleteReports']['shotsOnTargetFor'])
print('Independent arithmetic reconciled:', target, '/', shots)
print(json.dumps(result['counterObservations'], indent=2))

Independent arithmetic reconciled: 102 / 512
{
  "ownOnTargetEqualsGoalsInEveryCompleteReport": true,
  "opponentOnTargetEqualsGoalsInEveryCompleteReport": true,
  "reportsWithPositiveShotsButNoReportedShootCommand": [
    "live-02",
    "live-03",
    "live-04",
    "live-05",
    "live-07",
    "live-10",
    "live-13",
    "live-14",
    "live-16",
    "live-17",
    "live-18",
    "live-20",
    "live-21",
    "live-24",
    "live-25",
    "live-26",
    "live-30",
    "live-32",
    "live-35",
    "live-36",
    "live-37",
    "live-41",
    "live-43",
    "live-44",
    "live-47",
    "live-49",
    "live-52",
    "live-53",
    "live-55",
    "live-57",
    "live-59",
    "live-60",
    "live-63",
    "live-64",
    "live-65",
    "live-70",
    "live-75",
    "live-78",
    "live-79"
  ]
}


## Takeaways

The portal shows low conversion per recorded attempt, but distances and event definitions remain unavailable. Do not call 279 non-on-target attempts verified wide shots: blocked attempts and counter semantics are unresolved. Issued SHOOT commands and physical shots differ.

Provisional experiment: offer a shot only with confirmed possession, useful angle and a clear close opportunity; otherwise keep a safe passing outlet. In defence, retain central cover and use one nearby presser. Assess actual shot positions, later score changes and possession transitions before calibrating a distance cutoff. Current live Runtime errors must be fixed first.

Validation: notebook executes with local data only, without AWS calls. Source hash and coverage are printed above. A rerun against a newer checkout may include later reports; the creation summary is explicitly scoped to live49.